<a href="https://colab.research.google.com/github/bhavana-2005-stack/GENERATIVE_AI/blob/main/Vector_DB/ChromaDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install chromadb sentence-transformers groq

In [2]:
import os
import chromadb
from groq import Groq
from google.colab import userdata

#Setup

In [3]:
chroma_client=chromadb.PersistentClient(path="./chroma_db")
collection=chroma_client.get_or_create_collection(name="ABC_Handbook")
groq_client=Groq(api_key=userdata.get('grok_db'))

In [4]:
# --- Document: ABC Employee Handbook (chunked by section) ---
handbook_chunks = [
    "### Company Structure\nABC was founded in 2024 by Mark and John. The company operates in the AI education space with teams across engineering, content, and operations.",
    "### Work Hours\nStandard work hours are 9 AM to 6 PM, Monday through Friday. Flexible start times between 8–10 AM are permitted with manager approval.",
    "### Leave Policy\nEmployees are entitled to 12 casual leaves and 6 sick leaves per year. Leaves do not carry forward to the next year.",
    "### Remote Work\nEmployees must be present in the office on Tuesdays and Wednesdays. Remote work is permitted on all other working days.",
    "### Onboarding\nNew employees are expected to complete the onboarding module within two weeks of joining. The module covers company tools, processes, and code of conduct.",
    "### Learning and Development\nABC provides access to learning platforms. Priority courses are: Python for Data Science, SQL Fundamentals, and Machine Learning Foundations.",
]


In [5]:
collection.upsert(
    documents=handbook_chunks,
    ids=[f'chunk_{i}' for i in range(len(handbook_chunks))]
)
print(f'Indexed {len(handbook_chunks)} Chunks')

Indexed 6 Chunks


#RAG Query Function

#Build the RAG Prompt

In [6]:
def ask(question: str, top_k: int = 3) -> str:
    # Retrieve relevant chunks
    results = collection.query(
        query_texts=[question],
        n_results=top_k
    )

    retrieved = results["documents"][0]

    # Combine retrieved documents into context
    context = "\n\n".join(retrieved)

    # Build prompt
    prompt = f"""Answer the user question using only the context provided below.
If the answer cannot be found in the context, reply exactly: "I am not sure."
Do not use outside knowledge.

Context:
{context}

Question: {question}
"""

    # Query LLM
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [9]:
questions = [
    "How many casual leaves and sick leaved do employes get?",
    "Who found ABC",
    "What is the company's revenue last quarter?"
]
for q in questions:
  print(f"Question: {q}")
  print(f"Answer: {ask(q)}")
  print()

Question: How many casual leaves and sick leaved do employes get?
Answer: Employees are entitled to 12 casual leaves and 6 sick leaves per year.

Question: Who found ABC
Answer: Mark and John founded ABC.

Question: What is the company's revenue last quarter?
Answer: I am not sure.

